# f6_m02a_lime.ipynb

## Qué hace
Genera explicaciones locales con LIME (Local Interpretable Model-agnostic Explanations)
sobre los 3 perfiles representativos definidos en m01b (abandono, no abandono, riesgo).
LIME complementa SHAP: mientras SHAP usa valores de Shapley exactos basados en el modelo,
LIME aproxima localmente el comportamiento del modelo con un modelo lineal simple.
Modelo usado: CatBoost__balanced.

## Requisitos
- `results/fase6/perfiles_locales.parquet` (generado por f6_m01b)
- `data/05_modelado/X_test_prep.parquet`
- `data/05_modelado/X_train_prep.parquet` (necesario para LIME como referencia de distribución)
- `data/05_modelado/y_test.parquet`
- `data/05_modelado/models/CatBoost__balanced.pkl`
- Paquete: lime

## Genera
- `results/fase6/lime_abandono_real.png`
- `results/fase6/lime_no_abandono.png`
- `results/fase6/lime_zona_riesgo.png`
- `results/fase6/lime_explicaciones.parquet` — coeficientes LIME para los 3 perfiles
- `docs/html/fase6/m02a_lime.html`

## Flujo
Cargar datos + modelo + perfiles → instanciar LimeTabularExplainer →
explicar 3 perfiles → gráficos → comparativa SHAP vs LIME → HTML

## Siguiente
`f6_m03a_fairness.ipynb` — análisis de equidad algorítmica

In [1]:
# 1. CONFIGURACIÓN DE RUTAS
# ROOT detectado subiendo niveles hasta encontrar src/ — nunca hardcodeado
import sys
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

def _encontrar_root(start: Path) -> Path:
    for parent in [start] + list(start.parents):
        if (parent / 'src').is_dir():
            return parent
    raise FileNotFoundError('No se encontró src/ subiendo desde ' + str(start))

ROOT = _encontrar_root(Path.cwd())
sys.path.insert(0, str(ROOT))

DIR_DATA    = ROOT / 'data' / '05_modelado'
DIR_MODELS  = ROOT / 'data' / '05_modelado' / 'models'
DIR_RESULTS = ROOT / 'results' / 'fase6'
DIR_RESULTS.mkdir(parents=True, exist_ok=True)

print(f'ROOT:       {ROOT}')
print(f'DIR_MODELS: {DIR_MODELS}')
print(f'DIR_RESULTS:{DIR_RESULTS}')

ROOT:       C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI
DIR_MODELS: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\05_modelado\models
DIR_RESULTS:C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\results\fase6


In [2]:
# 2. IMPORTS
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import joblib
from lime import lime_tabular
from src.html.render import render_pagina_desde_fichero

matplotlib.rcParams['figure.dpi'] = 120

print('Imports OK.')

Imports OK.


In [3]:
# 3. CARGAR DATOS Y MODELO
# X_train_prep: LIME necesita los datos de entrenamiento como referencia
# de distribución para generar perturbaciones realistas.
# Los modelos de Fase 5 son sklearn.Pipeline — predict_proba funciona
# directamente sobre el Pipeline completo.

RUTA_PERFILES = DIR_RESULTS / 'perfiles_locales.parquet'
if not RUTA_PERFILES.exists():
    raise FileNotFoundError(
        'No se encontró perfiles_locales.parquet. '
        'Ejecuta primero f6_m01b_shap_local.ipynb.'
    )

X_test_prep  = pd.read_parquet(DIR_DATA / 'X_test_prep.parquet')
X_train_prep = pd.read_parquet(DIR_DATA / 'X_train_prep.parquet')
y_test       = pd.read_parquet(DIR_DATA / 'y_test.parquet').squeeze()
df_perfiles  = pd.read_parquet(RUTA_PERFILES)
pipeline_cat = joblib.load(DIR_MODELS / 'CatBoost__balanced.pkl')

feature_names = X_test_prep.columns.tolist()

# Detectar features categóricas — LIME necesita saberlo para perturbaciones correctas
cat_features = X_test_prep.select_dtypes(include=['object', 'category']).columns.tolist()
cat_idx = [feature_names.index(f) for f in cat_features]

print(f'X_test_prep:  {X_test_prep.shape}')
print(f'X_train_prep: {X_train_prep.shape}')
print(f'Features categóricas ({len(cat_features)}): {cat_features}')
print(f'Perfiles cargados: {df_perfiles["perfil"].tolist()}')

X_test_prep:  (6725, 19)
X_train_prep: (26896, 19)
Features categóricas (0): []
Perfiles cargados: ['abandono_real', 'no_abandono', 'zona_riesgo']


In [4]:
# 4. INSTANCIAR LIME EXPLAINER
# LimeTabularExplainer necesita los datos de entrenamiento para aprender
# la distribución de cada feature y generar perturbaciones realistas.
# mode='classification' porque predecimos abandono (0/1).
# discretize_continuous=True: LIME discretiza features numéricas en intervalos
# para que el modelo lineal local sea más interpretable.

explainer_lime = lime_tabular.LimeTabularExplainer(
    training_data=X_train_prep.values,
    feature_names=feature_names,
    categorical_features=cat_idx,
    class_names=['No abandona', 'Abandona'],
    mode='classification',
    discretize_continuous=True,
    random_state=42
)

print('LIME explainer instanciado.')
print(f'Datos de entrenamiento: {X_train_prep.shape[0]} observaciones')

LIME explainer instanciado.
Datos de entrenamiento: 26896 observaciones


In [5]:
# 5. GENERAR EXPLICACIONES LIME — 3 PERFILES
# Para cada perfil generamos una explicación con num_features=15.
# LIME crea ~5000 perturbaciones de la instancia, les aplica el modelo,
# y ajusta un modelo lineal local que explica el comportamiento en esa zona.
# num_samples=5000 es el estándar — más muestras = más estable pero más lento.
# Disk-first: si ya existen los resultados, los carga sin recalcular.

RUTA_LIME_EXP = DIR_RESULTS / 'lime_explicaciones.parquet'

# Recuperar índices de los perfiles desde perfiles_locales
# Los índices en perfiles_locales corresponden a posiciones en X_test_prep
prob = pipeline_cat.predict_proba(X_test_prep)[:, 1]
y_arr = y_test.values.ravel()

perfiles_orden = ['abandono_real', 'no_abandono', 'zona_riesgo']
perfiles_labels = {
    'abandono_real': 'Abandono (VP)',
    'no_abandono':   'No abandona (VN)',
    'zona_riesgo':   'Zona de riesgo',
}

# Recuperar índices posicionales en X_test_prep para cada perfil
# df_perfiles tiene el índice original de X_test — necesitamos posición iloc
idx_originales = df_perfiles.index.tolist()
idx_posicionales = [X_test_prep.index.get_loc(idx) for idx in idx_originales]

explicaciones = {}
registros = []

for i, (perfil, idx_pos) in enumerate(zip(perfiles_orden, idx_posicionales)):
    label = perfiles_labels[perfil]
    instancia = X_test_prep.iloc[idx_pos].values
    print(f'Calculando LIME para {label}...')

    exp = explainer_lime.explain_instance(
        data_row=instancia,
        predict_fn=pipeline_cat.predict_proba,
        num_features=15,
        num_samples=5000,
        labels=(1,)  # clase 1 = abandono
    )
    explicaciones[perfil] = exp

    # Guardar coeficientes para análisis posterior
    for feat, coef in exp.as_list(label=1):
        registros.append({
            'perfil':    perfil,
            'feature':   feat,
            'coef_lime': coef,
            'prob_real': prob[idx_pos],
            'y_real':    y_arr[idx_pos],
        })
    print(f'  Prob real: {prob[idx_pos]:.3f} | y_real: {y_arr[idx_pos]}')

df_lime = pd.DataFrame(registros)
df_lime.to_parquet(RUTA_LIME_EXP)
print(f'\nExplicaciones guardadas: {RUTA_LIME_EXP.name}')

Calculando LIME para Abandono (VP)...


  Prob real: 0.993 | y_real: 1
Calculando LIME para No abandona (VN)...


  Prob real: 0.040 | y_real: 0
Calculando LIME para Zona de riesgo...


  Prob real: 0.500 | y_real: 1

Explicaciones guardadas: lime_explicaciones.parquet


In [6]:
# 6. GRÁFICOS LIME — UNO POR PERFIL
# Barras horizontales: positivo (naranja) = empuja hacia abandono,
# negativo (azul) = reduce el riesgo. El valor es el coeficiente
# del modelo lineal local ajustado por LIME.

rutas_lime = {}
colores_perfil = {
    'abandono_real': '#e53e3e',
    'no_abandono':   '#38a169',
    'zona_riesgo':   '#dd6b20',
}

for perfil, idx_pos in zip(perfiles_orden, idx_posicionales):
    label = perfiles_labels[perfil]
    exp   = explicaciones[perfil]
    ruta  = DIR_RESULTS / f'lime_{perfil}.png'
    rutas_lime[perfil] = ruta

    # Extraer features y coeficientes
    feats_coefs = exp.as_list(label=1)
    feats = [fc[0] for fc in feats_coefs]
    coefs = [fc[1] for fc in feats_coefs]

    # Ordenar por valor absoluto
    orden = np.argsort(np.abs(coefs))
    feats = [feats[j] for j in orden]
    coefs = [coefs[j] for j in orden]

    colores_barras = ['#e53e3e' if c > 0 else '#3182ce' for c in coefs]

    fig, ax = plt.subplots(figsize=(10, 6))
    ax.barh(feats, coefs, color=colores_barras, alpha=0.8)
    ax.axvline(0, color='gray', linewidth=0.8)
    ax.set_xlabel('Coeficiente LIME (positivo = hacia abandono)')
    ax.set_title(
        f'LIME — {label} (prob={prob[idx_pos]:.3f})',
        fontsize=13, pad=12
    )
    plt.tight_layout()
    plt.savefig(ruta, dpi=120, bbox_inches='tight')
    plt.close()
    print(f'Gráfico guardado: {ruta.name}')

Gráfico guardado: lime_abandono_real.png


Gráfico guardado: lime_no_abandono.png


Gráfico guardado: lime_zona_riesgo.png


In [7]:
# 7. COMPARATIVA SHAP vs LIME — TOP FEATURES EN CONSENSO
# Cargamos importancias SHAP globales y comparamos con los coeficientes LIME
# del perfil de abandono real. Features que aparecen top en ambos métodos
# son las más robustas y defendibles ante el tribunal.

RUTA_SHAP_IMP = DIR_RESULTS / 'shap_importancia_comparativa.parquet'
if RUTA_SHAP_IMP.exists():
    df_shap_imp = pd.read_parquet(RUTA_SHAP_IMP)
    top10_shap  = df_shap_imp.nsmallest(10, 'rank_medio').index.tolist()

    # Top features LIME para perfil abandono_real (coef más positivos)
    lime_abandono = df_lime[df_lime['perfil'] == 'abandono_real'].copy()
    lime_abandono['feature_base'] = lime_abandono['feature'].str.extract(r'^([^<>=!]+)')[0].str.strip()
    top10_lime = lime_abandono.nlargest(10, 'coef_lime')['feature_base'].tolist()

    # Consenso: features que aparecen en ambos rankings
    consenso = [f for f in top10_shap if any(f in fl for fl in top10_lime)]

    print('Top 10 SHAP (consenso tres modelos):')
    for f in top10_shap:
        marca = '✅ consenso' if f in consenso else ''
        print(f'  {f:35s} {marca}')

    print(f'\nFeatures en consenso SHAP+LIME: {len(consenso)}/10')
    print(f'Consenso: {consenso}')
else:
    print('shap_importancia_comparativa.parquet no encontrado — omitiendo comparativa.')

Top 10 SHAP (consenso tres modelos):
  n_anios_beca                        ✅ consenso
  anios_sin_beca                      
  cred_superados_anio_1er             ✅ consenso
  situacion_laboral                   
  nota_1er_anio                       
  nota_acceso                         
  tuvo_beca                           
  edad_entrada                        ✅ consenso
  sexo                                
  max_pagos                           

Features en consenso SHAP+LIME: 3/10
Consenso: ['n_anios_beca', 'cred_superados_anio_1er', 'edad_entrada']


In [8]:
# 8. GENERAR HTML
# render_pagina_desde_fichero — estándar del proyecto.

import base64

def img_b64(ruta: Path) -> str:
    if not ruta.exists():
        return ''
    with open(ruta, 'rb') as f:
        return base64.b64encode(f.read()).decode()

def bloque_imagen(b64: str, titulo: str, caption: str) -> str:
    if not b64:
        return f'<p style="color:#e53e3e">Imagen no disponible: {titulo}</p>'
    return (
        '<div style="margin:24px 0">'
        f'<h3 style="color:#2d3748; font-size:15px">{titulo}</h3>'
        f'<img src="data:image/png;base64,{b64}" style="max-width:100%; border-radius:6px; box-shadow:0 2px 8px rgba(0,0,0,.1)">'
        f'<p style="color:#718096; font-size:12px; margin-top:6px">{caption}</p>'
        '</div>'
    )

contenido = (
    '<h2 style="color:#2d3748">LIME — Explicaciones Locales Alternativas</h2>'
    '<p style="color:#4a5568; font-size:14px; max-width:800px">'
    'LIME (<em>Local Interpretable Model-agnostic Explanations</em>) aproxima el comportamiento '
    'del modelo localmente con un modelo lineal simple. A diferencia de SHAP, que usa valores '
    'de Shapley exactos, LIME genera perturbaciones de la instancia y ajusta un modelo lineal '
    'en esa región. Ambos métodos son complementarios: el consenso entre SHAP y LIME '
    'identifica las variables más robustamente importantes.'
    '</p>'
    + bloque_imagen(
        img_b64(rutas_lime['abandono_real']),
        'LIME — Alumno que abandona (verdadero positivo)',
        'Barras naranjas/rojas: factores que el modelo lineal local identifica como '
        'impulsores del abandono. Barras azules: factores protectores. '
        'Los valores son coeficientes del modelo lineal local ajustado por LIME.')
    + bloque_imagen(
        img_b64(rutas_lime['no_abandono']),
        'LIME — Alumno que no abandona (verdadero negativo)',
        'Las barras azules dominan: LIME identifica factores protectores que '
        'reducen el riesgo predicho por el modelo.')
    + bloque_imagen(
        img_b64(rutas_lime['zona_riesgo']),
        'LIME — Alumno en zona de riesgo (probabilidad ~0.5)',
        'Factores de riesgo y protectores se equilibran. '
        'Este perfil es el más sensible a pequeños cambios en las variables de entrada.')
    + '<div style="margin-top:24px; padding:16px; background:#ebf8ff; '
    + 'border-left:4px solid #3182ce; border-radius:6px; font-size:13px; color:#2c5282">'
    + '<strong>SHAP vs LIME:</strong> SHAP calcula contribuciones exactas basadas en la '
    + 'teoría de juegos (valores de Shapley). LIME aproxima localmente con un modelo lineal. '
    + 'Cuando ambos métodos coinciden en las variables más importantes, '
    + 'el hallazgo es especialmente robusto y defendible.'
    + '</div>'
)

html_completo = render_pagina_desde_fichero('f6_m02a_lime.ipynb', contenido)
ruta_html = ROOT / 'docs' / 'html' / 'fase6' / 'm02a_lime.html'
ruta_html.parent.mkdir(parents=True, exist_ok=True)
ruta_html.write_text(html_completo, encoding='utf-8')
print(f'HTML generado: {ruta_html}')

HTML generado: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase6\m02a_lime.html
